# S02E03 — Failure

**Zadanie:** Pobrać ogromny plik logów z dnia awarii elektrowni, skompresować do **1500 tokenów**  
zawierając tylko zdarzenia istotne dla analizy, i przesłać technikom do weryfikacji.

**Techniki:** długi kontekst, filtrowanie danych, iteracja na podstawie feedbacku, zarządzanie tokenami

**Format odpowiedzi:** string z wierszami oddzielonymi `\n`, każdy wiersz = jedno zdarzenie

In [ ]:
import os, re, requests
from anthropic import Anthropic
from dotenv import load_dotenv, find_dotenv
import tiktoken

load_dotenv(find_dotenv())

API_KEY    = os.getenv("AI_DEVS_API_KEY")
client     = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
VERIFY_URL = "https://hub.ag3nts.org/verify"
LOG_URL    = f"https://hub.ag3nts.org/data/{API_KEY}/failure.log"
enc        = tiktoken.encoding_for_model("gpt-4o")

TOKEN_LIMIT = 1500


def count_tokens(text: str) -> int:
    return len(enc.encode(text))


print("Konfiguracja OK.")

In [ ]:
# Pobierz plik logów
resp = requests.get(LOG_URL)
resp.raise_for_status()
raw_log = resp.text
lines   = raw_log.splitlines()

print(f"Plik logów: {len(lines)} wierszy, {count_tokens(raw_log)} tokenów")
print("Przykładowe wiersze:")
for line in lines[:5]:
    print(" ", line)

In [ ]:
# Wstępne filtrowanie: zachowaj tylko WARN, ERROR, CRIT
critical_lines = [l for l in lines if any(lvl in l for lvl in ["WARN", "ERROR", "CRIT", "ERR"])]
critical_text  = "\n".join(critical_lines)

print(f"Po filtrze WARN/ERROR/CRIT: {len(critical_lines)} wierszy, {count_tokens(critical_text)} tokenów")
print("Przykłady:")
for l in critical_lines[:5]:
    print(" ", l)

In [ ]:
# Użyj Claude do kompresji logów do limitu tokenów
# Jeśli wstępnie przefiltrowane logi mieszczą się w limicie — bierzemy je.
# Jeśli nie — prosimy Claude o dalszą kompresję.

if count_tokens(critical_text) <= TOKEN_LIMIT:
    compressed_log = critical_text
    print(f"Wstępny filtr wystarczył: {count_tokens(compressed_log)} tokenów")
else:
    # Podziel logi na porcje, które Claude może przetworzyć
    CHUNK_SIZE = 3000  # tokenów na porcję
    chunks = []
    current_chunk = []
    current_tokens = 0

    for line in critical_lines:
        lt = count_tokens(line)
        if current_tokens + lt > CHUNK_SIZE:
            chunks.append("\n".join(current_chunk))
            current_chunk = [line]
            current_tokens = lt
        else:
            current_chunk.append(line)
            current_tokens += lt

    if current_chunk:
        chunks.append("\n".join(current_chunk))

    print(f"Porcje do przetworzenia: {len(chunks)}")

    # Zbierz streszczenia z każdej porcji
    summaries = []
    for i, chunk in enumerate(chunks):
        r = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=800,
            messages=[{
                "role": "user",
                "content": f"""Wybierz z tych logów TYLKO zdarzenia istotne dla analizy awarii elektrowni:
- podzespoły: zasilanie, chłodzenie, pompy, oprogramowanie, reaktor, turbiny, generatory
- poziomy: WARN, ERROR, CRIT
Zachowaj format: [YYYY-MM-DD HH:MM] [POZIOM] OPIS
Skracaj opisy do minimum. Jeden wiersz = jedno zdarzenie.\n\n{chunk}"""
            }]
        )
        summaries.append(r.content[0].text.strip())
        print(f"  Porcja {i+1}/{len(chunks)}: {count_tokens(summaries[-1])} tokenów")

    compressed_log = "\n".join(summaries)

    # Jeśli nadal za długie — dodatkowa runda kompresji
    if count_tokens(compressed_log) > TOKEN_LIMIT:
        r = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=1200,
            messages=[{
                "role": "user",
                "content": f"""Skompresuj te logi do maksimum {TOKEN_LIMIT} tokenów.
Zachowaj najważniejsze zdarzenia awarii. Skracaj opisy agresywnie.\n\n{compressed_log}"""
            }]
        )
        compressed_log = r.content[0].text.strip()

print(f"\nKońcowy log: {count_tokens(compressed_log)} tokenów, {len(compressed_log.splitlines())} wierszy")
print("Podgląd:")
print(compressed_log[:500])

In [ ]:
# Iteracyjne wysyłanie z uwzględnieniem feedbacku
MAX_ITERS = 5

for iteration in range(MAX_ITERS):
    tokens = count_tokens(compressed_log)
    print(f"\n=== Iteracja {iteration+1} ({tokens} tokenów) ===")

    if tokens > TOKEN_LIMIT:
        print(f"UWAGA: Log ma {tokens} tokenów, limit to {TOKEN_LIMIT}. Dalej kompresuję...")
        r = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=1200,
            messages=[{
                "role": "user",
                "content": f"Skompresuj agresywnie do max {TOKEN_LIMIT} tokenów:\n\n{compressed_log}"
            }]
        )
        compressed_log = r.content[0].text.strip()
        continue

    # Wyślij do Hub-u
    logs_str = compressed_log.replace("\n", "\\n")
    result = requests.post(VERIFY_URL, json={
        "apikey": API_KEY,
        "task": "failure",
        "answer": {"logs": compressed_log}
    }).json()

    print(f"Hub odpowiedział: code={result.get('code')}")
    print(f"Message: {result.get('message', '')}")

    if "FLG:" in str(result):
        flags = re.findall(r'\{FLG:[^}]+\}', str(result))
        print(f"\nFLAGA: {flags}")
        break

    if result.get("code") == 0:
        print("Sukces!")
        break

    # Popraw log na podstawie feedbacku
    feedback = result.get("message", "")
    if feedback:
        print(f"Feedback: {feedback}")
        r = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=1200,
            messages=[{
                "role": "user",
                "content": f"""Popraw logi na podstawie feedbacku techników. Limit: {TOKEN_LIMIT} tokenów.

Feedback: {feedback}

Dostępne logi źródłowe (WARN/ERROR/CRIT):
{critical_text[:8000]}

Obecna wersja:
{compressed_log}

Zwróć poprawioną wersję logów (tylko wiersze, bez komentarzy)."""
            }]
        )
        compressed_log = r.content[0].text.strip()

print("\nZakończono.")